In [ ]:
# ============================================================
# 6. Callbacks + Entraînement (ou CHARGEMENT d'un modèle existant)
# ----
# LOAD_EXISTING_MODEL = True  -> on charge best_model_densenet.keras et on saute l'entraînement
# LOAD_EXISTING_MODEL = False -> on relance l'entraînement en 2 phases (1-2h)
# ============================================================
LOAD_EXISTING_MODEL = True          # <-- mets False si tu veux ré-entraîner
MODEL_PATH = 'best_model_densenet_balanced.keras'

if LOAD_EXISTING_MODEL and os.path.exists(MODEL_PATH):
    print(f"Chargement du modele existant : {MODEL_PATH}")
    # compile=False car focal est une fonction custom, on recompile apres
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal,
        metrics=['accuracy']
    )
    # Historique vide (on n affichera pas les courbes d apprentissage)
    history = type('H', (), {})()
    history.history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    print("Modele charge. Entrainement saute.")
    print(f"   Parametres totaux : {model.count_params():,}")

else:
    print("Entrainement complet (2 phases)...")

    early_stop = EarlyStopping(monitor='val_loss', patience=6,
                               restore_best_weights=True, verbose=1)
    reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=3, min_lr=1e-7, verbose=1)
    checkpoint = ModelCheckpoint(MODEL_PATH,
                                 monitor='val_loss', save_best_only=True, verbose=1)

    # --- Poids de classes ---
    labels  = train_generator.classes
    weights = class_weight.compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels)
    dict_weights = {0: weights[0], 1: weights[1]}
    print(f"Poids classes : {dict_weights}")

    # --- PHASE 1 ---
    print("\n" + "="*60)
    print("PHASE 1 - Entrainement de la tete (backbone gele)")
    print("="*60)
    model.compile(
        optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss=focal, metrics=['accuracy']
    )
    history_1 = model.fit(
        train_generator, epochs=10,
        validation_data=validation_generator,
        class_weight=dict_weights,
        callbacks=[early_stop, reduce_lr, checkpoint]
    )

    # --- PHASE 2 ---
    print("\n" + "="*60)
    print("PHASE 2 - Fine-tuning des derniers blocs")
    print("="*60)
    base_model.trainable = True
    for layer in base_model.layers[:-40]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal, metrics=['accuracy']
    )
    history_2 = model.fit(
        train_generator, epochs=20,
        validation_data=validation_generator,
        class_weight=dict_weights,
        callbacks=[early_stop, reduce_lr, checkpoint]
    )

    # Fusion des historiques
    history = type('H', (), {})()
    history.history = {
        k: history_1.history[k] + history_2.history[k]
        for k in history_1.history.keys()
    }

In [ ]:
# ============================================================
# 7. Fine-tuning équilibré (warm start + oversampling + focal pondéré)
# ----
# DO_BALANCED_FINETUNE = True  -> exécute le warm start (~20-40 min)
# DO_BALANCED_FINETUNE = False -> charge best_model_densenet_balanced.keras s'il existe, sinon garde le modèle actuel
# ============================================================
import shutil

DO_BALANCED_FINETUNE = False            # <-- passe à False après la première exécution réussie
BALANCED_MODEL_PATH  = 'best_model_densenet_balanced.keras'
BALANCED_DIR         = 'train_balanced'
OVERSAMPLE_FACTOR    = 3               # NORMAL dupliqué x3 → ~4023 vs 3875 PNEUMONIA

if DO_BALANCED_FINETUNE:

    # ---------- 7.1 Construction du dataset rééquilibré (symlinks) ----------
    src_normal    = os.path.join(train_dir, 'NORMAL')
    src_pneumonia = os.path.join(train_dir, 'PNEUMONIA')
    dst_normal    = os.path.join(BALANCED_DIR, 'NORMAL')
    dst_pneumonia = os.path.join(BALANCED_DIR, 'PNEUMONIA')

    if os.path.exists(BALANCED_DIR):
        shutil.rmtree(BALANCED_DIR)
    os.makedirs(dst_normal)
    os.makedirs(dst_pneumonia)

    def link_or_copy(src, dst):
        try:
            os.symlink(os.path.abspath(src), dst)
        except OSError:
            shutil.copy(src, dst)

    # PNEUMONIA : 1x (original)
    for f in os.listdir(src_pneumonia):
        if f.startswith('.'):
            continue
        link_or_copy(os.path.join(src_pneumonia, f), os.path.join(dst_pneumonia, f))

    # NORMAL : x OVERSAMPLE_FACTOR (noms différents pour éviter collision)
    for dup in range(OVERSAMPLE_FACTOR):
        for f in os.listdir(src_normal):
            if f.startswith('.'):
                continue
            dst_name = f if dup == 0 else f'dup{dup}_{f}'
            link_or_copy(os.path.join(src_normal, f), os.path.join(dst_normal, dst_name))

    n_norm = len([x for x in os.listdir(dst_normal) if not x.startswith('.')])
    n_pneu = len([x for x in os.listdir(dst_pneumonia) if not x.startswith('.')])
    print(f"Dataset rééquilibré : {n_norm} NORMAL | {n_pneu} PNEUMONIA | ratio {n_pneu/n_norm:.2f}:1")

    # ---------- 7.2 Nouveau générateur sur le dataset équilibré ----------
    train_generator_balanced = train_datagen.flow_from_directory(
        BALANCED_DIR,
        target_size=(224, 224),
        batch_size=BATCH,
        class_mode='categorical',
        color_mode='rgb'
    )

    # ---------- 7.3 Focal loss pondéré (alpha plus élevé sur NORMAL) ----------
    def categorical_focal_loss_weighted(gamma=2.0, alpha=(0.7, 0.3)):
        alpha_t = tf.constant(alpha, dtype=tf.float32)
        def loss(y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            ce = -y_true * tf.math.log(y_pred)
            weight = alpha_t * tf.math.pow(1.0 - y_pred, gamma)
            return tf.reduce_sum(weight * ce, axis=-1)
        return loss

    focal_weighted = categorical_focal_loss_weighted(gamma=2.0, alpha=(0.7, 0.3))

    # ---------- 7.4 Recompile en warm start (LR très bas) ----------
    model.compile(
        optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4),
        loss=focal_weighted,
        metrics=['accuracy']
    )

    # ---------- 7.5 Callbacks ----------
    checkpoint_bal = ModelCheckpoint(
        BALANCED_MODEL_PATH, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_bal = EarlyStopping(
        monitor='val_loss', patience=4,
        restore_best_weights=True, verbose=1
    )
    reduce_bal = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-8, verbose=1
    )

    # ---------- 7.6 Fine-tuning court (warm start) ----------
    print("\n" + "="*60)
    print("WARM START - Fine-tuning équilibré (~8 epochs, 20-40 min)")
    print("="*60)
    history_balanced = model.fit(
        train_generator_balanced,
        epochs=8,
        validation_data=validation_generator,
        callbacks=[checkpoint_bal, early_bal, reduce_bal]
    )

    print(f"\nModele equilibre sauvegarde : {BALANCED_MODEL_PATH}")

elif os.path.exists(BALANCED_MODEL_PATH):
    print(f"Chargement du modele equilibre existant : {BALANCED_MODEL_PATH}")
    model = tf.keras.models.load_model(BALANCED_MODEL_PATH, compile=False)
    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal,
        metrics=['accuracy']
    )
    print("Modele equilibre charge. La suite du notebook utilisera ce modele.")

else:
    print("Aucun modele equilibre disponible. Le notebook continuera avec le modele initial.")

In [ ]:
model.summary()

In [ ]:
# 8. Evaluation du modele sur le jeu de test
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Creer un generateur d'evaluation SANS shuffle
eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_xray)
eval_generator = eval_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

# Predictions (softmax retourne 2 probabilites)
y_pred_prob = model.predict(eval_generator)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = eval_generator.classes

# Noms des classes
class_names = list(eval_generator.class_indices.keys())
print("Classes:", eval_generator.class_indices)
print("\n" + "="*50)
print("RAPPORT DE CLASSIFICATION")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))